# 01 · Train the player / ball / hoop detector

One YOLO detector for every on-court object the pipeline needs: `Player`, `Ref`, `Ball`, `Hoop`
(plus `Clock`, `Scoreboard`, `Overlay`, which the pipeline ignores but which help the model learn
what *isn't* a player).

Changes from the original training notebooks:

* **The dataset downloads in YOLO format** (`"yolov8"`). The player notebook asked for `"yolov5x-seg.pt"`,
  which is a weights file name, not a format.
* **Base weights exist.** `yolov5.pt` is not a downloadable checkpoint, which is why that run failed.
  We use `yolo11m.pt` (swap in `yolo11l.pt` / `yolo11x.pt` on an A100).
* **Runs save to Google Drive** with `save_period`, so a Colab disconnect (the ball run stopped at
  epoch 244 of 250) resumes instead of starting over.
* **API key from Colab Secrets**, not hard-coded.
* **Per-class validation**, because "all classes mAP" hides a weak `Ball` class, and the ball is the
  hardest and most important object.
* **Optional ball-only model** trained at higher resolution for small, blurred balls.
* **More data, better measured** (optional cell after the download): merge extra YOLO datasets
  with a class rename map, add hard-negative frames (crowd, bench, replays), fold in frames you
  labelled from your own clips (notebook 03), and re-split so validation has ~20% of the images
  and of the balls. With 32 validation images one missed ball moved ball recall by 3.6 points.

In [ ]:
# Runtime > Change runtime type > GPU before running.
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

# Everything that must survive a Colab disconnect (runs, weights) lives in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
!git clone -q {REPO_URL} /content/courtvision || (cd /content/courtvision && git pull -q)
!pip -q install -e "/content/courtvision[models,train]"

## Dataset

In [ ]:
# Store your Roboflow key in Colab's Secrets panel (key icon on the left) as
# ROBOFLOW_API_KEY. Never paste it into a cell: notebooks get shared and committed.
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("workspace-5ujvu").project("basketball-players-fy4c2-vfsuv")
dataset = project.version(17).download("yolov8", location="/content/datasets/players-v17")
DATA_ROOT = dataset.location
DATA_YAML = f"{DATA_ROOT}/data.yaml"
print(open(DATA_YAML).read())

## Optional: grow and re-split the data

The ball has 213 training boxes, and 23% of the detector's false detections are phantom balls,
which turn into false possession changes. More labelled balls and more "nothing here" frames are
the biggest lever. Three sources, all optional:

* `EXTRA_DATASETS`: other YOLO-format basketball datasets (Roboflow Universe has several; check that
  their frames look like broadcast footage). Map their class names onto ours, `None` drops a class.
* `NEGATIVES_DIR`: a Drive folder of frames with nothing to label (crowd, bench, timeouts, replays,
  ads). They get empty labels and teach the model what is *not* a player or ball.
* `OWN_LABELLED`: your own NBA broadcast frames, exported from notebook 03, corrected in Roboflow
  and downloaded in YOLO format. These are worth more per image than anything else, because they
  are the footage the pipeline actually runs on.

The re-split pools train and valid and puts 20% of the ball images, and 20% of the rest, into
validation. Leave everything empty and set `RESPLIT = True` to only fix the validation size.

In [ ]:
import shutil, yaml
from courtvision.training.datasets import class_counts, merge_datasets, resplit

EXTRA_DATASETS = [
    # ("workspace", "project", version, {"basketball": "Ball", "person": "Player", "rim": "Hoop"}),
]
NEGATIVES_DIR = f"{DRIVE_ROOT}/training_data/negatives"     # skipped if missing
OWN_LABELLED = f"{DRIVE_ROOT}/training_data/own_labelled"    # Roboflow YOLO export of notebook 03 frames
RESPLIT = True

names = yaml.safe_load(open(DATA_YAML))["names"]
names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
sources = [(DATA_ROOT, None)]
for ws, proj, ver, rename in EXTRA_DATASETS:
    extra = rf.workspace(ws).project(proj).version(ver).download("yolov8", location=f"/content/datasets/{proj}-v{ver}")
    sources.append((extra.location, rename))
if os.path.exists(f"{OWN_LABELLED}/data.yaml"):
    sources.append((OWN_LABELLED, None))
negatives = NEGATIVES_DIR if os.path.isdir(NEGATIVES_DIR) else None

if len(sources) > 1 or negatives or RESPLIT:
    MERGED = "/content/datasets/detector-merged"
    shutil.rmtree(MERGED, ignore_errors=True)
    print("boxes per class:", merge_datasets(sources, names, MERGED, negatives=negatives))
    if RESPLIT:
        print("images per split:", resplit(MERGED, val_frac=0.2, stratify_class=names.index("Ball")))
    DATA_ROOT, DATA_YAML = MERGED, f"{MERGED}/data.yaml"
for split, counts in class_counts(DATA_ROOT).items():
    print(f"{split:5s}", counts)

In [ ]:
# Label counts per split. If a class has fewer than ~100 validation boxes its mAP is noise;
# set RESPLIT = True above.
import collections, glob, yaml
names = yaml.safe_load(open(DATA_YAML))["names"]
for split in ["train", "valid", "test"]:
    counts = collections.Counter()
    files = glob.glob(f"{DATA_ROOT}/{split}/labels/*.txt")
    for f in files:
        for line in open(f):
            counts[names[int(line.split()[0])]] += 1
    print(f"{split:5s} {len(files):5d} images  ", dict(counts))

## Train

Broadcast frames are wide and the ball is ~15-25 px, so train at `imgsz=1280` and keep mosaic on
(it creates many small-object examples). `fliplr` is fine here: left/right does not change a
player's class. `batch=-1` lets Ultralytics pick the largest batch that fits the GPU.

In [ ]:
from ultralytics import YOLO

# New name per data change: an existing finished run would otherwise be "resumed" and stop at once.
RUN_NAME = "detector_yolo11m_1280_v2"
last = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/last.pt"
resume = os.path.exists(last)
model = YOLO(last if resume else "yolo11m.pt")
model.train(
    data=DATA_YAML, imgsz=1280, epochs=150, patience=30, batch=-1,
    cos_lr=True, close_mosaic=15, save_period=10,
    project=f"{DRIVE_ROOT}/runs", name=RUN_NAME, exist_ok=True, resume=resume,
)

## Validate per class

In [ ]:
best = YOLO(f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt")
# Ultralytics calls the validation split "val" (Roboflow's folder is valid/, mapped by data.yaml).
metrics = best.val(data=DATA_YAML, imgsz=1280, split="val")
present = list(metrics.box.ap_class_index)  # classes with validation labels
for cls_id, name in best.names.items():
    if cls_id in present:
        p, r, ap50, ap = metrics.box.class_result(present.index(cls_id))
        print(f"{name:12s} P={p:.3f} R={r:.3f} mAP50={ap50:.3f} mAP50-95={ap:.3f}")
    else:
        print(f"{name:12s} no validation labels")

In [ ]:
import shutil
shutil.copy(f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt", f"{DRIVE_ROOT}/models/player_detector.pt")
print("Saved", f"{DRIVE_ROOT}/models/player_detector.pt")

## Optional: dedicated ball detector

If `Ball` recall above is below ~0.85, train a ball-only model. One class means all capacity goes to
the ball. Point
`models.ball_detector` in `configs/default.yaml` at the result; the pipeline then uses it for the
ball and the main model for everything else.

The first ball-only run (YOLO11s from COCO weights, 1536 px) stopped at epoch 70 with NaN
validation losses and precision swinging between 0 and 0.94: 213 balls are too few to learn a
detector from scratch. This version starts from **your trained detector** instead, whose backbone
already knows broadcast basketball, so the ball head is all that has to adapt. It trains at 1280:
on a real 1080p broadcast the median ball is 49 px, about 33 px at 1280, which is plenty.
Only replace `ball_detector.pt` if ball recall below beats the main detector's `Ball` row.

In [ ]:
import pathlib, shutil, yaml
src = pathlib.Path(DATA_ROOT)
dst = pathlib.Path("/content/datasets/ball-only")
ball_id = [i for i, n in enumerate(names) if n.lower() == "ball"][0]
for split in ["train", "valid", "test"]:
    if not (src / split).exists():
        continue
    (dst / split / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copytree(src / split / "images", dst / split / "images", dirs_exist_ok=True)
    for f in (src / split / "labels").glob("*.txt"):
        keep = [" ".join(["0"] + l.split()[1:]) for l in open(f) if int(l.split()[0]) == ball_id]
        (dst / split / "labels" / f.name).write_text("\n".join(keep) + ("\n" if keep else ""))
yaml.safe_dump({"path": str(dst), "train": "train/images", "val": "valid/images",
                "test": "test/images", "names": {0: "Ball"}}, open(dst / "data.yaml", "w"))
print(open(dst / "data.yaml").read())

In [ ]:
BALL_RUN = "ball_from_detector_1280"
BALL_BASE = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt"   # or "yolo11s.pt" for a small model from COCO
last = f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/last.pt"
resume = os.path.exists(last)
ball_model = YOLO(last if resume else BALL_BASE)
ball_model.train(data=str(dst / "data.yaml"), imgsz=1280, epochs=150, patience=40, batch=-1,
                 cos_lr=True, close_mosaic=20, save_period=10, lr0=0.002, warmup_epochs=1,
                 project=f"{DRIVE_ROOT}/runs", name=BALL_RUN, exist_ok=True, resume=resume)
ball_best = YOLO(f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/best.pt")
bm = ball_best.val(data=str(dst / "data.yaml"), imgsz=1280, split="val")
print(f"ball-only P={bm.box.mp:.3f} R={bm.box.mr:.3f} mAP50={bm.box.map50:.3f}")

In [ ]:
# Keep the dedicated model only if it beats the main detector's Ball row printed earlier.
shutil.copy(f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/best.pt", f"{DRIVE_ROOT}/models/ball_detector.pt")